## Importando as bibliotecas

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from astropy.io import fits

import warnings
warnings.filterwarnings("ignore")

## Geneva-Copenhagen survey

### Nordstrom+, 2004
* Paper: [The Geneva-Copenhagen survey of the Solar neighbourhood. Ages, metallicities, and kinematic properties of ~14 000 F and G dwarfs](https://www.aanda.org/articles/aa/abs/2004/18/aa0959/aa0959.html)
* Vizier: [Geneva-Copenhagen Survey of Solar neighbourhood : V/117A](https://cdsarc.cds.unistra.fr/viz-bin/cat/V/117A#/browse)

### Holmberg+, 2009 (Geneva-Copenhagen survey III)
* Paper: [The Geneva-Copenhagen survey of the solar neighbourhood III. Improved distances, ages, and kinematics](https://www.aanda.org/articles/aa/full_html/2009/27/aa11191-08/aa11191-08.html)
* Vizier: [Geneva-Copenhagen Survey of Solar neighbourhood III : V/130](https://cdsarc.cds.unistra.fr/viz-bin/cat/V/130#/browse)

### Casagrande+, 2011 (Geneva-Copenhagen survey re-analysis)
* Paper: [New constraints on the chemical evolution of the solar neighbourhood and Galactic disc(s)](https://www.aanda.org/articles/aa/full_html/2011/06/aa16276-10/aa16276-10.html)
* Vizier: [Geneva-Copenhagen survey re-analysis : J/A+A/530/A138](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/530/A138#/browse)

In [2]:
DATA_PATH_CASAGRANDE = "../data/gcs-casagrande2011.fits"
DATA_PATH_GCSI = "../data/gcs-nordstrom2004.fits"
DATA_PATH_GCSIII = "../data/gcs-holmberg2009.fits"

# Lendo os arquivos fits dos catálogos
fits_table_casagrande = fits.open(DATA_PATH_CASAGRANDE)[1].data
fits_table_GCSI = fits.open(DATA_PATH_GCSI)[1].data
fits_table_GCSIII = fits.open(DATA_PATH_GCSIII)[1].data

# Transformando em pandas.DataFrame
casagrande = pd.DataFrame.from_records(fits_table_casagrande)
gcsi = pd.DataFrame.from_records(fits_table_GCSI)
gcsiii = pd.DataFrame.from_records(fits_table_GCSIII)

Para visualizar a descrição dos parâmetros, você pode utilizar o [ReadMe](https://cdsarc.cds.unistra.fr/ftp/J/A+A/530/A138/ReadMe) do Geneva-Copenhagen survey re-analysis (Casagrande+, 2011).

Note que a versão do artigo de Casagrande et al. (2011) não apresenta os valores da **magnitude absoluta** e $v\sin i$, mas possui outras informações, como o fluxo bolométrico, a correção por avermelhamento interestelar e a velocidade radial. Para atender aos objetivos do artigo e garantir a completude da amostra, será realizado um primeiro cruzamento de dados com as versões I e III do Geneva-Copenhagen Survey. A versão III fornece valores de magnitude absoluta e da distância, mas não de $v\sin i$, enquanto a versão I, embora obsoleta, contém informações sobre este último (consulte o [ReadMe da versão I](https://cdsarc.cds.unistra.fr/ftp/V/117A/ReadMe) e o [ReadMe da versão III](https://cdsarc.cds.unistra.fr/ftp/V/130/ReadMe) para mais detalhes).

**Observação importante:** O Geneva-Copenhagen Survey I também possui dados de magnitude absoluta e distância, mas por ser uma versão *obsoleta*, consideramos utilizar os valores da versão III.

Para adicionar apenas as colunas necessárias em nosso conjunto de dados, criaremos um novo dataframe `df`. Selecionaremos as colunas `vsini`(velocidade de rotação) do `gcsi` e as colunas `VMag` (magnitude absoluta), `e_VMag` (erro padrão da magnitude absoluta) e `Dist` (Distância) do `gcsiii`.

In [3]:
# Aqui vamos remover os espaços em branco do nome das estrelas para evitar erros
casagrande["Name"] = casagrande["Name"].str.strip()
gcsi["Name"] = gcsi["Name"].str.strip()
gcsiii["Name"] = gcsiii["Name"].str.strip()

# Vamos considerar apenas as colunas desejadas em cada DataFrmae
gcsi_selecionado = gcsi[["Name", "vsini"]]
gcsiii_selecionado = gcsiii[["Name", "VMag", "e_VMag", "Dist"]]

# A nossa chave primária é o nome da estrela (Name), vamos juntar os DataFrames
df = pd.merge(casagrande, gcsi_selecionado, on="Name")
df = pd.merge(df, gcsiii_selecionado, on="Name")

# Vamos eliminar duplicadas que pode ter surgido ao fazer o merge
df = df.drop_duplicates(subset="Name", keep="first")

# Veja como ficou (Dataframe CasaGrande + GCSI[VSINI] + GCSIII[VMAG, EVMAG, DIST])
df.head()

,pedig,HIP,Name,m_Name,plx,e_plx,logg,Teff,e_Teff,Fbol,...,M.EB,M.5B,M.16B,M.MB,M.84B,M.95B,vsini,VMag,e_VMag,Dist
0,irfm,420,HD 23,,23.86,0.67,4.44,6203.0,81.0,2.720080e-08,...,1.12,1.05,1.08,1.12,1.16,1.17,4,4.44,0.06,42
1,irfm,459,HD 67,,19.54,0.88,4.58,5746.0,67.0,8.285120e-09,...,0.97,0.90,0.93,0.97,1.00,1.02,4,5.28,0.10,51
2,irfm,493,HD 101,,26.93,0.56,4.41,5957.0,77.0,2.831090e-08,...,1.02,0.94,0.97,1.02,1.07,1.09,2,4.61,0.05,37
3,irfm,490,HD 105,,25.39,0.59,4.43,6035.0,62.0,2.641430e-08,...,1.05,0.98,1.01,1.05,1.10,1.12,15,4.53,0.05,39
4,irfm,530,HD 153,,9.38,0.71,3.91,5869.0,85.0,1.223690e-08,...,1.22,1.14,1.17,1.21,1.26,1.31,5,3.22,0.16,107


In [4]:
print(f"Quantidade de estrelas: {len(df)} \nQuantidade de parâmetros: {len(df.columns)}")

Quantidade de estrelas: 16539 
Quantidade de parâmetros: 74


No código acima, foi necessário remover os espaços em branco presentes nas strings da coluna `Name`. Caso contrário, uma estrela com o nome "$\text{HD}\,1$" não seria identificada como correspondente a outra tabela, caso o seu nome estivesse registrado como "$\text{HD}\,1\quad$". Após esse procedimento, as colunas restantes foram combinadas em um único dataframe denominado `df`.

## Filtragem (PARTE 1)

Vamos iniciar o processo de filtragem. Antes de explorar as filtragens mencionadas no artigo, devemos levar em consideração a filtragem indicada pelo próprio catálogo do Casagrande (ver [ReadMe Nota 1](https://cdsarc.cds.unistra.fr/ftp/J/A+A/530/A138/ReadMe)):

* Stars with **irfm** pedigree have best quality photometry, are likely to be single and non-variable (see Section 2 in the paper). For these stars, effective temperatures and bolometric fluxes have been computed via InfraRed Flux Method directly.

* **clbr** identifies all remaining stars, for which effective temperatures and bolometric fluxes have been obtained using the colour-metallicity-temperature and colour-metallicity-flux calibrations in Casagrande et al. (2010, Cat. J/A+A/512/A54). 

* *NOTICE* that some clbr stars might include double or multiple components (often recognizable by their Name e.g. HD XXXXX/X or HD XXXX AB), for which matching the closest Tycho2 and 2MASS photometry might be more uncertain. **Therefore in some cases the photometry might result in a mix of the two sources and so are the derived parameters, which should not be used**. It is responsibility of the user to take extra care in identifying cases likely affected by this issue.

Dessa forma, nos próximos passos, *consideraremos apenas as estrelas com `pedig == irfm` e eliminaremos quaisquer duplicatas de estrelas.*

In [5]:
# Fazendo um cópia do nosso DataFrame "original"
df_versao1 = df.copy()

# Eliminando pedig == clbr (isso evita estrelas duplas e com componentes múltiplos)
df_versao1 = df_versao1[df_versao1["pedig"] == "irfm"]

In [6]:
print(f"Quantidade de estrelas com pedig == irfm: {len(df_versao1)}")

Quantidade de estrelas com pedig == irfm: 6670


Vamos excluir as colunas `pedig` e `m_Name` da nossa amostra:

In [7]:
# A coluna pedig agora só possui valores == irfm e m_Name é uma coluna vazia
# você pode verificar isso fazendo:
# df_versao1["pedig"].value_counts()
# df_versao1["m_Name"].value_counts()
df_versao1 = df_versao1.drop(["pedig", "m_Name"], axis=1)

Podemos também excluir aquelas estrelas que possuem valores nulos para a velocidade de rotação ($v\sin i$) e distância:

In [8]:
df_versao1 = df_versao1[df_versao1["vsini"] != 0]
df_versao1 = df_versao1[df_versao1["Dist"] != 0]

In [9]:
print(f"Quantidade de estrelas: {len(df_versao1)} \nQuantidade de parâmetros: {len(df_versao1.columns)}")

Quantidade de estrelas: 5183 
Quantidade de parâmetros: 72


## Filtragem (PARTE 2)

A partir deste momento, vamos começar a tomar algumas decisões que foram abordadas no artigo original (seção 2). Primeiramente, é necessário encontrar as coordenadas X, Y e Z como funções da longitude e latitude galáctica:
\begin{align}
X &= R\cos(b)\cos(l),  \\
Y &= R\cos(b)\sin(l), \\
Z &= R\sin(b),
\end{align}
onde $l$ é a longitude e $b$ latitude galáctica.

In [10]:
# Fazendo uma nova cópia porque vamos modificar novamente
df_versao2 = df_versao1.copy()

# Agora vamos usar as equações (1) a (3)
df_versao2["X"] = df_versao2["Dist"] * np.cos(np.radians(df_versao2["GLAT"])) * np.cos(np.radians(df_versao2["GLON"]))
df_versao2["Y"] = df_versao2["Dist"] * np.cos(np.radians(df_versao2["GLAT"])) * np.sin(np.radians(df_versao2["GLON"]))
df_versao2["Z"] = df_versao2["Dist"] * np.sin(np.radians(df_versao2["GLAT"]))

In [11]:
# Vamos usar o identificador para poder encontrar o tipo espectral 
# e o ramo no diagrama HR em:
# https://simbad.cds.unistra.fr/simbad/sim-fid
nome_arquivo = '../data/stars_names.txt'

with open(nome_arquivo, 'w') as arquivo:
    for nome in df_versao2['Name']:
        arquivo.write(nome + '\n')

print(f"Os nomes foram gravados em '{nome_arquivo}'.")

Os nomes foram gravados em '../data/stars_names.txt'.


Utilizando a lista de indentificadores anteriormente, podemos usar no [SIMBAD](https://simbad.cds.unistra.fr/simbad/sim-fid), para encontrar o tipo espectral e conseguir definir aquelas que estão na sequência principal:

In [12]:
SIMBAD = '../data/simbad.csv'
simbad = pd.read_csv(SIMBAD, delimiter=';', skiprows=[1], na_values="~")
simbad = simbad.drop(simbad.columns[0], axis=1)
simbad.columns = simbad.columns.str.strip()
simbad.head()

,typed ident,identifier,typ,"coord1 (ICRS,J2000/2000)",Mag U,Mag B,Mag V,Mag R,Mag I,spec. type,#bib,#not
0,HD 23,HD 23,PM*,00 05 07.4950976136 -52 09 06.260110200,~,8.11,7.52,~,~,G0V,46.0,0.0
1,HD 67,HD 67,PM*,00 05 28.4520372600 -61 13 33.074149188,~,9.47,8.80,~,~,G5V,49.0,0.0
2,HD 101,HD 101,PM*,00 05 54.7486323216 +18 14 06.026302140,8.050,8.010,~,7.1,~,F8,58.0,0.0
3,HD 105,HD 105,PM*,00 05 52.5447817152 -41 45 11.044988352,~,8.13,7.53,~,~,G0V,178.0,0.0
4,HD 153,HD 153,PM*,00 06 26.0245156200 +42 45 09.885919032,~,8.97,8.36,~,~,G1V,38.0,0.0


In [13]:
# Vamos ver o que temos:
simbad["typ"].value_counts()

typ
PM*    3533
*      1424
SB*     131
BY*      22
**       20
EB*      16
Er*       9
Pe*       7
V*        5
Ro*       4
dS*       3
EB?       1
gD*       1
El*       1
RS*       1
Pu*       1
Em*       1
TT*       1
Ir*       1
SB?       1
Name: count, dtype: int64

De acordo com a [lista dos tipos de objetos](https://simbad.cds.unistra.fr/guide/otypes.htx) que também está salvo em `data/otypes.list`, podemos ver que nossa amostra tem:

| Object Type | Description                |
|-------------|----------------------------|
| PM*         | High Proper Motion Star   |
| *           | Star                       |
| SB*         | Spectroscopic Binary      |
| BY*         | BY Dra Variable            |
| **          | Double or Multiple Star    |
| EB*         | Eclipsing Binary           |
| Er*         | Eruptive Variable          |
| Pe*         | Chemically Peculiar Star   |
| V*          | Variable Star              |
| Ro*         | Rotating Variable          |
| dS*         | delta Sct Variable         |
| EB?         | Eclipsing Binary           |
| gD*         | gamma Dor Variable         |
| El*         | Ellipsoidal Variable       |
| RS*         | RS CVn Variable            |
| Pu*         | Pulsating Variable         |
| Em*         | Emission-line Star         |
| TT*         | T Tauri Star               |
| Ir*         | Irregular Variable         |
| SB?         | Spectroscopic Binary       |


In [14]:
# Vamos juntar essa tabela (só o queremos) com nosso dados
simbad_selecionado = simbad[["identifier", "typ", "spec. type"]]
simbad_selecionado.columns = ["Name", "ObjType", "spec_type"]

simbad_selecionado["Name"] = simbad_selecionado["Name"].str.strip()
simbad_selecionado["ObjType"] = simbad_selecionado["ObjType"].str.strip()
simbad_selecionado["spec_type"] = simbad_selecionado["spec_type"].str.strip()

# A nossa chave primária é o nome da estrela (Name), vamos juntar os DataFrames
df_versao3 = pd.merge(df_versao2, simbad_selecionado, on="Name")

# Vamos eliminar duplicadas que pode ter surgido ao fazer o merge
df_versao3 = df_versao3.drop_duplicates(subset="Name", keep="first")

# Veja como ficou
df_versao3.head()

,HIP,Name,plx,e_plx,logg,Teff,e_Teff,Fbol,[Fe/H],[M/H],...,M.95B,vsini,VMag,e_VMag,Dist,X,Y,Z,ObjType,spec_type
0,420,HD 23,23.86,0.67,4.44,6203.0,81.0,2.720080e-08,-0.07,-0.05,...,1.17,4,4.44,0.06,42,14.129955,-12.300312,-37.590513,PM*,G0V
1,459,HD 67,19.54,0.88,4.58,5746.0,67.0,8.285120e-09,-0.09,-0.05,...,1.02,4,5.28,0.10,51,19.770410,-21.530308,-41.792065,PM*,G5V
2,493,HD 101,26.93,0.56,4.41,5957.0,77.0,2.831090e-08,-0.24,-0.18,...,1.09,2,4.61,0.05,37,-8.319715,25.605449,-25.379978,PM*,F8
3,490,HD 105,25.39,0.59,4.43,6035.0,62.0,2.641430e-08,-0.18,-0.14,...,1.12,15,4.53,0.05,39,10.320716,-5.361183,-37.225536,PM*,G0V
4,530,HD 153,9.38,0.71,3.91,5869.0,85.0,1.223690e-08,0.08,0.11,...,1.31,5,3.22,0.16,107,-41.384108,92.083817,-35.453154,PM*,G1V


In [15]:
# De acordo com a tabela anterior, vamos considerar apenas aquelas PM* e *
df_versao4 = df_versao3[(df_versao3["ObjType"] == "PM*") | (df_versao3["ObjType"] == "*")]

In [16]:
# Vamos separar nossos datasets
tipo_F = df_versao4[df_versao4['spec_type'].str.startswith('F')].reset_index(drop=True)
tipo_G = df_versao4[df_versao4['spec_type'].str.startswith('G')].reset_index(drop=True)

In [17]:
print(f"Quantidade de estrelas: {len(df_versao4)}")
print(f"Quantidade de estrelas do tipo F: {len(tipo_F)}")
print(f"Quantidade de estrelas do tipo G: {len(tipo_G)}")

Quantidade de estrelas: 4910
Quantidade de estrelas do tipo F: 2869
Quantidade de estrelas do tipo G: 1884


In [18]:
tipo_F

,HIP,Name,plx,e_plx,logg,Teff,e_Teff,Fbol,[Fe/H],[M/H],...,M.95B,vsini,VMag,e_VMag,Dist,X,Y,Z,ObjType,spec_type
0,493,HD 101,26.93,0.56,4.41,5957.0,77.0,2.831090e-08,-0.24,-0.18,...,1.09,2,4.61,0.05,37,-8.319715,25.605449,-25.379978,PM*,F8
1,547,HD 189,5.70,0.89,3.79,6374.0,65.0,1.041110e-08,0.13,0.14,...,1.67,9,2.84,0.28,140,18.016135,17.655026,-137.708819,PM*,F7V
2,556,HD 200,9.03,0.83,3.87,5998.0,88.0,1.404450e-08,-0.21,-0.14,...,1.35,5,3.00,0.20,111,-4.857308,46.923101,-100.477008,PM*,F7/8V
3,606,HD 268,19.90,0.64,4.24,6486.0,74.0,3.951410e-08,-0.24,-0.18,...,1.31,9,3.56,0.07,50,6.608911,5.670527,-49.235835,PM*,F5V
4,596,HD 276,12.52,0.52,4.11,6611.0,135.0,2.485580e-08,-0.17,-0.11,...,1.45,7,3.04,0.09,80,36.161888,-49.354711,-51.540570,PM*,F2V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2864,67,HD 224806,6.65,0.70,3.71,6489.0,107.0,1.997670e-08,0.00,0.04,...,1.78,28,1.95,0.23,150,-36.939558,112.484623,-92.101458,PM*,F5
2865,90,HD 224842,8.83,0.65,3.84,6393.0,76.0,2.260380e-08,-0.00,0.03,...,1.63,30,2.38,0.16,113,31.523936,-14.975408,-107.475479,PM*,F5IV/V
2866,195,HD 224999,8.50,0.76,3.78,6377.0,111.0,2.463730e-08,0.05,0.07,...,1.69,22,2.27,0.19,118,-47.181820,103.963509,-29.823896,*,F8
2867,238,HD 225045,16.35,0.47,3.79,6217.0,80.0,8.193480e-08,0.30,0.29,...,1.72,17,2.33,0.06,61,6.326580,12.363221,-59.398023,PM*,F6/7V


Nossa amostra completa consiste em 4910 estrelas, das quais 2869 são do tipo F e 1884 são do tipo G. Em comparação com o artigo original, onde foram relatadas 6615 estrelas, observamos uma diminuição no número de estrelas do tipo F (de 3671 para 2869) e para o tipo G (de 2816 para 1884).

In [19]:
# Vamos salvar nossos dados para estrelas do tipo F e G
tipo_F.to_csv("../data/gcs-Fstars.csv", index=False)
tipo_G.to_csv("../data/gcs-Gstars.csv", index=False)